# PersianBookCoverLens: Persian Book Cover Title Recognition

This notebook implements an end-to-end Persian book-cover title recognition pipeline for the final assignment of the **Vision-Language Models (VLM)** course.

The project supports two execution tracks:

1. **CPU-compatible OCR fine-tuning track** — the default track for free Google Colab sessions without GPU access. It trains a lightweight image-to-text OCR model on Persian book-cover images and compares metrics before and after fine-tuning.
2. **Qwen VLM track** — an optional track for GPU runtimes, designed for zero-shot/fine-tuning experiments with Qwen2.5-VL.

The default configuration is intentionally set to the CPU-compatible track so the notebook can run in a standard free Colab environment without crashing.

## 1. Installation

Run this cell once in Google Colab. If Colab asks you to restart the runtime after installation, restart and continue from the next section. The default CPU track uses only lightweight dependencies plus PyTorch, which is already available in Colab.

> For the optional Qwen/VLM track, set `INSTALL_QWEN_DEPS = True` before running this cell. Qwen inference/fine-tuning requires a GPU runtime and is not recommended on CPU-only Colab.

In [ ]:
import sys
import subprocess

INSTALL_QWEN_DEPS = False  # Keep False for CPU-only Colab final run.
REPAIR_PIL = False         # Set True only if you hit a PIL/Pillow import error.

BASE_PACKAGES = [
    "pandas==2.2.2",
    "Pillow==11.3.0",
    "datasets>=3.0.0",
    "rapidfuzz>=3.9.0",
    "matplotlib>=3.8.0",
    "tqdm>=4.66.0",
]

QWEN_PACKAGES = [
    "transformers>=4.53.0",
    "accelerate>=1.8.0",
    "peft>=0.13.0",
    "bitsandbytes>=0.43.0",
    "qwen-vl-utils>=0.0.8",
]

if REPAIR_PIL:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pillow", "PIL"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall", "Pillow==11.3.0"], check=True)

packages = BASE_PACKAGES + (QWEN_PACKAGES if INSTALL_QWEN_DEPS else [])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--upgrade-strategy", "only-if-needed", *packages], check=True)
print("Installation finished. If this was the first install in Colab, restart the runtime once, then continue from Section 2.")

## 2. Imports, Reproducibility, and Global Configuration

The default values below are designed for a CPU-only Colab session.

For a final CPU run:

```python
EXPERIMENT_TRACK = "cpu_light_ocr"
FAST_DEV_RUN = False
CPU_SAFE_MODE = True
```

For a very quick smoke test:

```python
FAST_DEV_RUN = True
```

For the optional Qwen/VLM track, use a GPU runtime and set:

```python
EXPERIMENT_TRACK = "qwen_vlm"
CPU_SAFE_MODE = False
```

In [ ]:
import os
import re
import json
import math
import random
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_NAME = "PersianBookCoverLens"
DATASET_ID = "shenasa/bookroom-persian-book-covers-and-titles"
QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# Default final mode for CPU-only Colab.
EXPERIMENT_TRACK = "cpu_light_ocr"  # Options: "cpu_light_ocr" or "qwen_vlm"
FAST_DEV_RUN = False
CPU_SAFE_MODE = True

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if (HAS_CUDA and not CPU_SAFE_MODE) else "cpu"

if FAST_DEV_RUN:
    TRAIN_LIMIT = 24
    EVAL_LIMIT = 8
    EPOCHS = 1
else:
    # CPU-safe final run. Increase if you have enough time.
    TRAIN_LIMIT = 200
    EVAL_LIMIT = 50
    EPOCHS = 3

# Assignment-scale settings for the Qwen/GPU track.
if EXPERIMENT_TRACK == "qwen_vlm" and not FAST_DEV_RUN:
    TRAIN_LIMIT = 1000
    EVAL_LIMIT = 200
    EPOCHS = 1

IMAGE_WIDTH = 128
IMAGE_HEIGHT = 160
MAX_TITLE_CHARS = 64
BATCH_SIZE = 16 if DEVICE == "cpu" else 4
LEARNING_RATE = 2e-3

print({
    "project": PROJECT_NAME,
    "track": EXPERIMENT_TRACK,
    "device": DEVICE,
    "cuda_available": HAS_CUDA,
    "fast_dev_run": FAST_DEV_RUN,
    "train_limit": TRAIN_LIMIT,
    "eval_limit": EVAL_LIMIT,
    "epochs": EPOCHS,
})

## 3. Persian Text Normalization and Evaluation Metrics

The assignment requires **Exact Match**. This notebook also reports two stronger diagnostic metrics:

- **Levenshtein Similarity**: robust to small OCR errors.
- **Word-level F1**: measures title overlap at word level.

Both raw and normalized exact match are calculated.

In [ ]:
PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
ARABIC_DIGITS = "٠١٢٣٤٥٦٧٨٩"
EN_DIGITS = "0123456789"
DIGIT_TRANSLATION = str.maketrans(PERSIAN_DIGITS + ARABIC_DIGITS, EN_DIGITS + EN_DIGITS)

DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670\u0640]")
PUNCT_RE = re.compile(r"[\s\u200c\-ـ_.,،؛:!؟?()\[\]{}<>«»\"'`~|/\\]+")


def normalize_persian_text(text: str) -> str:
    """Normalize Persian/Arabic text for fair OCR comparison."""
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(DIGIT_TRANSLATION)
    text = text.replace("ي", "ی").replace("ى", "ی").replace("ك", "ک")
    text = text.replace("ۀ", "ه").replace("ة", "ه")
    text = DIACRITICS_RE.sub("", text)
    text = PUNCT_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def word_f1(pred: str, truth: str) -> float:
    pred_words = normalize_persian_text(pred).split()
    true_words = normalize_persian_text(truth).split()
    if not pred_words and not true_words:
        return 1.0
    if not pred_words or not true_words:
        return 0.0
    pred_counts = {}
    for w in pred_words:
        pred_counts[w] = pred_counts.get(w, 0) + 1
    true_counts = {}
    for w in true_words:
        true_counts[w] = true_counts.get(w, 0) + 1
    overlap = sum(min(pred_counts.get(w, 0), true_counts.get(w, 0)) for w in set(pred_counts) | set(true_counts))
    precision = overlap / max(len(pred_words), 1)
    recall = overlap / max(len(true_words), 1)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def levenshtein_similarity(pred: str, truth: str) -> float:
    p = normalize_persian_text(pred)
    t = normalize_persian_text(truth)
    denom = max(len(p), len(t), 1)
    return 1.0 - (Levenshtein.distance(p, t) / denom)


def compute_metrics(rows: List[Dict]) -> Dict[str, float]:
    """Compute title-recognition metrics from prediction rows."""
    if len(rows) == 0:
        return {
            "n": 0,
            "exact_match_raw": 0.0,
            "exact_match_normalized": 0.0,
            "levenshtein_similarity": 0.0,
            "word_level_f1": 0.0,
        }
    raw_exact = []
    norm_exact = []
    lev_scores = []
    f1_scores = []
    for r in rows:
        pred = str(r.get("prediction", ""))
        truth = str(r.get("ground_truth", ""))
        raw_exact.append(float(pred.strip() == truth.strip()))
        norm_exact.append(float(normalize_persian_text(pred) == normalize_persian_text(truth)))
        lev_scores.append(levenshtein_similarity(pred, truth))
        f1_scores.append(word_f1(pred, truth))
    return {
        "n": len(rows),
        "exact_match_raw": float(np.mean(raw_exact)),
        "exact_match_normalized": float(np.mean(norm_exact)),
        "levenshtein_similarity": float(np.mean(lev_scores)),
        "word_level_f1": float(np.mean(f1_scores)),
    }

# Minimal sanity check for metrics.
test_rows = [{"prediction": "شاهنامه فردوسی", "ground_truth": "شاهنامه‌ی فردوسی"}]
compute_metrics(test_rows)

## 4. Load Dataset and Select Subsets

The course assignment asks for:

- a subset of the training split, e.g. 1,000 examples;
- the first 200 examples from the test split.

Because this notebook defaults to a CPU-safe final run, the default CPU track uses 200 training samples and 50 test samples. The Qwen/GPU track uses 1,000/200 by default.

In [ ]:
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)

train_full = raw_dataset["train"]
test_full = raw_dataset["test"]

train_data = train_full.select(range(min(TRAIN_LIMIT, len(train_full))))
test_data = test_full.select(range(min(EVAL_LIMIT, len(test_full))))

print(f"Selected train samples: {len(train_data)}")
print(f"Selected test samples: {len(test_data)}")
print("Columns:", train_data.column_names)
print("Example title:", train_data[0]["text"])

## 5. Visual Inspection

This cell displays a few book-cover samples with their Persian ground-truth titles.

In [ ]:
def show_samples(dataset, n=4):
    n = min(n, len(dataset))
    plt.figure(figsize=(4 * n, 5))
    for i in range(n):
        sample = dataset[i]
        img = sample["image"]
        title = sample["text"]
        ax = plt.subplot(1, n, i + 1)
        ax.imshow(img)
        ax.set_title(normalize_persian_text(title)[:40], fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_data, n=4)

## 6. CPU-compatible Fine-tuning Track

This is the default track for CPU-only Colab.

It trains a small image-to-character model directly on book-cover images. This is not intended to outperform full VLMs; its purpose is to provide a **fully reproducible before/after fine-tuning experiment** under strict hardware limits.

The model architecture is intentionally small:

- compact CNN image encoder;
- GRU character decoder;
- greedy decoding for title generation.

This track produces the required files:

- `outputs/baseline_predictions.csv`
- `outputs/finetuned_predictions.csv`
- `outputs/metrics_summary.csv`
- `outputs/metrics_comparison.png`

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]
BASE_PERSIAN_CHARS = list(
    "ءآاأإبپتثجچحخدذرزژسشصضطظعغفقکكگگلمنوهةهیيىئؤ "
    "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
    "-–—_.,،؛:!؟?()[]{}«»/\\'\""
)


def build_char_vocab(dataset) -> Tuple[Dict[str, int], Dict[int, str]]:
    chars = set(BASE_PERSIAN_CHARS)
    for sample in dataset:
        chars.update(list(normalize_persian_text(sample["text"])))
    chars = sorted(chars)
    vocab = SPECIAL_TOKENS + [c for c in chars if c not in SPECIAL_TOKENS]
    char2id = {c: i for i, c in enumerate(vocab)}
    id2char = {i: c for c, i in char2id.items()}
    return char2id, id2char

char2id, id2char = build_char_vocab(train_data)
PAD_ID = char2id["<pad>"]
SOS_ID = char2id["<sos>"]
EOS_ID = char2id["<eos>"]
UNK_ID = char2id["<unk>"]
VOCAB_SIZE = len(char2id)
print("Vocabulary size:", VOCAB_SIZE)


def encode_title(title: str, max_len: int = MAX_TITLE_CHARS):
    text = normalize_persian_text(title)[: max_len - 1]
    token_ids = [char2id.get(ch, UNK_ID) for ch in text] + [EOS_ID]
    decoder_input = [SOS_ID] + token_ids[:-1]
    labels = token_ids
    pad_len = max_len - len(labels)
    if pad_len > 0:
        decoder_input += [PAD_ID] * pad_len
        labels += [-100] * pad_len
    else:
        decoder_input = decoder_input[:max_len]
        labels = labels[:max_len]
    return torch.tensor(decoder_input, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


def decode_ids(ids: List[int]) -> str:
    chars = []
    for idx in ids:
        idx = int(idx)
        if idx == EOS_ID:
            break
        if idx in (PAD_ID, SOS_ID):
            continue
        chars.append(id2char.get(idx, ""))
    return normalize_persian_text("".join(chars))


def preprocess_image(img: Image.Image, augment: bool = False) -> torch.Tensor:
    img = img.convert("RGB")
    img = ImageOps.contain(img, (IMAGE_WIDTH, IMAGE_HEIGHT), method=Image.Resampling.BICUBIC)
    canvas = Image.new("RGB", (IMAGE_WIDTH, IMAGE_HEIGHT), color=(255, 255, 255))
    left = (IMAGE_WIDTH - img.width) // 2
    top = (IMAGE_HEIGHT - img.height) // 2
    canvas.paste(img, (left, top))
    img = canvas

    if augment:
        # Light augmentation only. Strong transforms can damage OCR text.
        if random.random() < 0.35:
            img = img.rotate(random.uniform(-2.0, 2.0), fillcolor=(255, 255, 255))
        if random.random() < 0.35:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))
        if random.random() < 0.25:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))

    arr = np.asarray(img).astype("float32") / 255.0
    arr = (arr - 0.5) / 0.5
    tensor = torch.tensor(arr).permute(2, 0, 1)
    return tensor


class BookCoverTitleDataset(Dataset):
    def __init__(self, hf_dataset, augment: bool = False):
        self.data = hf_dataset
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = preprocess_image(sample["image"], augment=self.augment)
        decoder_input, labels = encode_title(sample["text"])
        return {
            "image": image,
            "decoder_input_ids": decoder_input,
            "labels": labels,
            "ground_truth": sample["text"],
        }


class TinyCoverOCR(nn.Module):
    """A compact image-to-character OCR model for CPU-compatible fine-tuning."""
    def __init__(self, vocab_size: int, emb_dim: int = 64, hidden_dim: int = 128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.context = nn.Linear(64, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.decoder = nn.GRU(emb_dim + hidden_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def encode(self, images):
        features = self.encoder(images).flatten(1)
        context = torch.tanh(self.context(features))
        return context

    def forward(self, images, decoder_input_ids):
        context = self.encode(images)
        embeddings = self.embedding(decoder_input_ids)
        repeated_context = context.unsqueeze(1).expand(-1, embeddings.size(1), -1)
        decoder_inputs = torch.cat([embeddings, repeated_context], dim=-1)
        outputs, _ = self.decoder(decoder_inputs)
        logits = self.output(outputs)
        return logits

    @torch.no_grad()
    def generate(self, image_tensor, max_len: int = MAX_TITLE_CHARS):
        self.eval()
        if image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
        image_tensor = image_tensor.to(next(self.parameters()).device)
        context = self.encode(image_tensor)
        token = torch.tensor([[SOS_ID]], dtype=torch.long, device=image_tensor.device)
        hidden = None
        generated = []
        for _ in range(max_len):
            emb = self.embedding(token[:, -1:])
            repeated_context = context.unsqueeze(1)
            decoder_input = torch.cat([emb, repeated_context], dim=-1)
            out, hidden = self.decoder(decoder_input, hidden)
            logits = self.output(out[:, -1, :])
            next_id = int(torch.argmax(logits, dim=-1).item())
            if next_id == EOS_ID:
                break
            generated.append(next_id)
            token = torch.cat([token, torch.tensor([[next_id]], dtype=torch.long, device=image_tensor.device)], dim=1)
        return decode_ids(generated)

### 6.1 Baseline Evaluation Before Fine-tuning

The baseline here is the randomly initialized lightweight OCR model before training. This gives a strict before/after measurement under CPU constraints.

In [ ]:
def predict_cpu_model(model: nn.Module, dataset, stage: str) -> List[Dict]:
    rows = []
    model.eval()
    for i in tqdm(range(len(dataset)), desc=f"Predicting ({stage})"):
        sample = dataset[i]
        image = preprocess_image(sample["image"], augment=False)
        pred = model.generate(image)
        rows.append({
            "index": i,
            "stage": stage,
            "prediction": pred,
            "ground_truth": sample["text"],
            "ground_truth_normalized": normalize_persian_text(sample["text"]),
        })
    return rows

if EXPERIMENT_TRACK == "cpu_light_ocr":
    train_torch = BookCoverTitleDataset(train_data, augment=True)
    test_torch = BookCoverTitleDataset(test_data, augment=False)

    model = TinyCoverOCR(vocab_size=VOCAB_SIZE).to(DEVICE)

    baseline_rows = predict_cpu_model(model, test_data, stage="before_finetuning")
    baseline_metrics = compute_metrics(baseline_rows)

    pd.DataFrame(baseline_rows).to_csv(OUTPUT_DIR / "baseline_predictions.csv", index=False, encoding="utf-8-sig")
    print("Baseline metrics:")
    print(json.dumps(baseline_metrics, ensure_ascii=False, indent=2))
else:
    print("Skipping CPU OCR baseline because EXPERIMENT_TRACK is not 'cpu_light_ocr'.")

### 6.2 Fine-tune the CPU-compatible OCR Model

This cell performs real training on CPU. It is intentionally small and stable for free Colab.

If the run is too slow, reduce `TRAIN_LIMIT` or `EPOCHS` in Section 2.

In [ ]:
def train_cpu_model(model: nn.Module, train_dataset: Dataset):
    model.train()
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}")
        for batch in pbar:
            images = batch["image"].to(DEVICE)
            decoder_input_ids = batch["decoder_input_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images, decoder_input_ids)
            loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            losses.append(float(loss.item()))
            pbar.set_postfix({"loss": np.mean(losses[-20:])})

        epoch_loss = float(np.mean(losses))
        history.append({"epoch": epoch, "train_loss": epoch_loss})
        print(f"Epoch {epoch}: train_loss={epoch_loss:.4f}")
    return history

if EXPERIMENT_TRACK == "cpu_light_ocr":
    history = train_cpu_model(model, train_torch)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

    torch.save({
        "model_state_dict": model.state_dict(),
        "char2id": char2id,
        "id2char": id2char,
        "config": {
            "image_width": IMAGE_WIDTH,
            "image_height": IMAGE_HEIGHT,
            "max_title_chars": MAX_TITLE_CHARS,
            "vocab_size": VOCAB_SIZE,
        },
    }, OUTPUT_DIR / "tiny_cover_ocr_finetuned.pt")

    print("Saved fine-tuned CPU OCR checkpoint to outputs/tiny_cover_ocr_finetuned.pt")
else:
    print("Skipping CPU OCR fine-tuning because EXPERIMENT_TRACK is not 'cpu_light_ocr'.")

### 6.3 Evaluation After Fine-tuning

This cell evaluates the fine-tuned model on the same test subset and compares the metrics before and after fine-tuning.

In [ ]:
def save_metrics_and_plot(metrics_before: Dict[str, float], metrics_after: Dict[str, float]):
    metrics_df = pd.DataFrame([
        {"stage": "before_finetuning", **metrics_before},
        {"stage": "after_finetuning", **metrics_after},
    ])
    metrics_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False, encoding="utf-8-sig")

    plot_metrics = ["exact_match_normalized", "levenshtein_similarity", "word_level_f1"]
    plot_df = metrics_df.set_index("stage")[plot_metrics]
    ax = plot_df.T.plot(kind="bar", figsize=(9, 5))
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Before vs After Fine-tuning")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "metrics_comparison.png", dpi=160)
    plt.show()
    return metrics_df

if EXPERIMENT_TRACK == "cpu_light_ocr":
    finetuned_rows = predict_cpu_model(model, test_data, stage="after_finetuning")
    finetuned_metrics = compute_metrics(finetuned_rows)

    pd.DataFrame(finetuned_rows).to_csv(OUTPUT_DIR / "finetuned_predictions.csv", index=False, encoding="utf-8-sig")
    metrics_df = save_metrics_and_plot(baseline_metrics, finetuned_metrics)

    print("Metrics summary:")
    display(metrics_df)

    print("Sample predictions after fine-tuning:")
    display(pd.DataFrame(finetuned_rows).head(10))
else:
    print("Skipping CPU OCR evaluation because EXPERIMENT_TRACK is not 'cpu_light_ocr'.")

## 7. Optional Qwen2.5-VL Track for GPU Runtime

This section is included to keep the project aligned with the original VLM requirement. It should only be used when a GPU is available.

The CPU-compatible track above is the practical final run for CPU-only Colab. The Qwen track is the stronger VLM direction for a GPU environment.

> Do not run this section on CPU-only Colab. It will be too slow or may run out of memory.

In [ ]:
if EXPERIMENT_TRACK == "qwen_vlm":
    if not torch.cuda.is_available():
        raise RuntimeError("Qwen/VLM track requires a GPU runtime. Use the CPU track or enable GPU.")

    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    from qwen_vl_utils import process_vision_info

    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )
    qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)

    def qwen_predict_title(image: Image.Image, max_new_tokens: int = 64) -> str:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Read the Persian book title from this cover. Return only the title text."},
                ],
            }
        ]
        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = qwen_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(qwen_model.device)

        generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = qwen_processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        return output_text.strip()

    qwen_rows = []
    for i in tqdm(range(len(test_data)), desc="Qwen zero-shot evaluation"):
        sample = test_data[i]
        pred = qwen_predict_title(sample["image"])
        qwen_rows.append({
            "index": i,
            "stage": "qwen_zero_shot",
            "prediction": pred,
            "ground_truth": sample["text"],
            "ground_truth_normalized": normalize_persian_text(sample["text"]),
        })

    pd.DataFrame(qwen_rows).to_csv(OUTPUT_DIR / "qwen_zero_shot_predictions.csv", index=False, encoding="utf-8-sig")
    print(json.dumps(compute_metrics(qwen_rows), ensure_ascii=False, indent=2))
else:
    print("Qwen/VLM track is disabled. Set EXPERIMENT_TRACK='qwen_vlm' and use GPU to run this section.")

## 8. Error Analysis

This section prints the weakest examples according to Levenshtein similarity. It is useful for the final report and presentation.

In [ ]:
def attach_error_scores(csv_path: Path):
    df = pd.read_csv(csv_path)
    df["prediction_normalized"] = df["prediction"].fillna("").apply(normalize_persian_text)
    df["levenshtein_similarity"] = df.apply(lambda r: levenshtein_similarity(r["prediction"], r["ground_truth"]), axis=1)
    df["word_level_f1"] = df.apply(lambda r: word_f1(r["prediction"], r["ground_truth"]), axis=1)
    return df.sort_values("levenshtein_similarity", ascending=True)

finetuned_path = OUTPUT_DIR / "finetuned_predictions.csv"
if finetuned_path.exists():
    error_df = attach_error_scores(finetuned_path)
    display(error_df.head(10))
else:
    print("No fine-tuned predictions found yet. Run the CPU track or Qwen track first.")

## 9. Final Report Checklist

Use these artifacts in your final submission:

- Notebook: this file
- Baseline predictions: `outputs/baseline_predictions.csv`
- Fine-tuned predictions: `outputs/finetuned_predictions.csv`
- Metrics table: `outputs/metrics_summary.csv`
- Comparison chart: `outputs/metrics_comparison.png`
- Fine-tuned CPU checkpoint: `outputs/tiny_cover_ocr_finetuned.pt`

Recommended wording for CPU-only execution:

> The original VLM fine-tuning track is implemented for GPU runtimes. Since the available Colab runtime was CPU-only, I added a CPU-compatible OCR fine-tuning track to demonstrate the required before/after fine-tuning comparison in a reproducible way. The same evaluation metrics are used in both tracks.